# Chapter 11 — SVM & Ensembles
**Track:** ML from Scratch · California Housing — regression benchmark + classification comparison

## Core Idea

**SVM:** of all separating hyperplanes, pick the one with the largest margin (gap to the nearest training points). Kernel trick extends this to non-linear boundaries without computing the high-dimensional feature map explicitly.

**Bagging (Random Forest):** train 100–500 independent trees on bootstrapped data. Average predictions → variance drops by $\sim 1/N$ (minus a floor set by inter-tree correlation $\rho$).

$$\text{Var}(\text{ensemble}) = \rho\sigma^2 + \frac{1-\rho}{N}\sigma^2$$

**Boosting (XGBoost):** train trees sequentially, each fitting the residuals of the previous ensemble → bias drops with each round.

```
Random Forest: parallel trees → variance reduction
XGBoost:       sequential trees → bias reduction (+ regularisation)
SVM:           single maximum-margin boundary
```

## Running Example

**Regression:** predict California district median house value from all 8 features.  
We benchmark Linear Regression (Ch.1 baseline) → Decision Tree → Random Forest → XGBoost.

**Classification:** high-value district? — adds SVM to the Ch.10 comparison.

Same dataset throughout — differences in RMSE and R² are purely the model, not the data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (mean_squared_error, r2_score,
                              f1_score, roc_auc_score, accuracy_score)

# ── Data ──────────────────────────────────────────────────────────────────────
data          = fetch_california_housing()
X, y_reg      = data.data, data.target
feature_names = list(data.feature_names)
y_cls         = (y_reg > np.median(y_reg)).astype(int)

(X_train, X_test,
 y_tr, y_te,
 y_tr_cls, y_te_cls) = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42)

scaler    = StandardScaler()
X_tr_sc   = scaler.fit_transform(X_train)
X_te_sc   = scaler.transform(X_test)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Regression target range: [{y_reg.min():.2f}, {y_reg.max():.2f}] $100k")
print(f"Classification balance:  {y_cls.mean():.3f} positive")

## SVM — Maximum Margin Classifier

The hard-margin SVM minimises $\frac{1}{2}\|\mathbf{w}\|^2$ subject to $y_i(\mathbf{w}^\top\mathbf{x}_i + b) \geq 1$.

The soft-margin version (C-SVM) adds slack: the $C$ parameter trades margin width against training violations.
The RBF kernel $K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2)$ maps data to an infinite-dimensional feature space.

In [ ]:
# SVM on 2D projection (MedInc vs Latitude) for decision boundary visualisation
feat_idx = [0, 6]   # MedInc, Latitude
feat_lbls = [feature_names[i] for i in feat_idx]

X_2d_tr = X_tr_sc[:, feat_idx]
X_2d_te = X_te_sc[:, feat_idx]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, C_val in zip(axes, [0.1, 1.0, 100.0]):
    svm_2d = SVC(kernel='rbf', C=C_val, gamma='scale', random_state=42)
    svm_2d.fit(X_2d_tr, y_tr_cls)

    # Decision boundary grid
    x_min, x_max = X_2d_tr[:, 0].min() - 0.5, X_2d_tr[:, 0].max() + 0.5
    y_min, y_max = X_2d_tr[:, 1].min() - 0.5, X_2d_tr[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150),
                          np.linspace(y_min, y_max, 150))
    Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, cmap='coolwarm')
    ax.scatter(X_2d_te[:, 0], X_2d_te[:, 1],
               c=y_te_cls, cmap='coolwarm', s=4, alpha=0.4)
    # Mark support vectors
    sv = svm_2d.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=30, facecolors='none',
               edgecolors='black', linewidths=0.8, label=f'SVs={len(sv)}')
    f1 = f1_score(y_te_cls, svm_2d.predict(X_2d_te))
    ax.set_title(f'C={C_val}  F1={f1:.3f}', fontweight='bold')
    ax.set_xlabel(feat_lbls[0]); ax.set_ylabel(feat_lbls[1])
    ax.legend(fontsize=8); ax.grid(alpha=0.2)

plt.suptitle('SVM RBF Kernel — Decision Boundary vs C  (2-feature projection)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Full 8-feature SVM
svm_full = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
svm_full.fit(X_tr_sc, y_tr_cls)

y_pred_svm = svm_full.predict(X_te_sc)
print(f"SVM (rbf, C=10) on all 8 features:")
print(f"  Test accuracy: {accuracy_score(y_te_cls, y_pred_svm):.4f}")
print(f"  Test F1:       {f1_score(y_te_cls, y_pred_svm):.4f}")
print(f"  Test AUC-ROC:  {roc_auc_score(y_te_cls, svm_full.predict_proba(X_te_sc)[:, 1]):.4f}")
print(f"  Support vectors: {svm_full.n_support_} per class  "
      f"({svm_full.support_vectors_.shape[0]} total of {len(X_train):,} training points)")

## Random Forest — Bagging in Action

Each tree sees a bootstrapped subset of training rows and a random subset of features at each split.  
The OOB (out-of-bag) examples — the ~37% not sampled into each tree — give a free validation estimate.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_tr)   # tree-based: no scaling needed

y_pred_rf = rf.predict(X_test)
rmse_rf   = np.sqrt(mean_squared_error(y_te, y_pred_rf))
r2_rf     = r2_score(y_te, y_pred_rf)

print(f"Random Forest (200 trees):")
print(f"  Test RMSE: {rmse_rf:.4f}  R²: {r2_rf:.4f}")
print(f"  OOB  R²:   {rf.oob_score_:.4f}  (free validation — no held-out set used)")

In [ ]:
# Feature importances — averaged over all 200 trees (more stable than a single DT)
rf_importances = rf.feature_importances_
sorted_idx     = np.argsort(rf_importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh([feature_names[i] for i in sorted_idx[::-1]],
        rf_importances[sorted_idx[::-1]],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Importance (mean Gini reduction across 200 trees)')
ax.set_title('Random Forest Feature Importances (stable across seeds)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("RF importances (averaged over 200 trees — much more stable than single DT):")
for i in sorted_idx:
    print(f"  {feature_names[i]:12s}: {rf_importances[i]:.4f}  {'█' * int(rf_importances[i] * 60)}")

## Variance Reduction — How Many Trees Are Enough?

In [ ]:
n_trees_range  = [1, 5, 10, 25, 50, 100, 200, 300, 400, 500]
rmse_by_ntrees = []

for n in n_trees_range:
    m = RandomForestRegressor(n_estimators=n, max_features='sqrt',
                               random_state=42, n_jobs=-1)
    m.fit(X_train, y_tr)
    preds = m.predict(X_test)
    rmse_by_ntrees.append(np.sqrt(mean_squared_error(y_te, preds)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_trees_range, rmse_by_ntrees, 'o-', color='steelblue', linewidth=2)
ax.axhline(rmse_by_ntrees[-1], color='coral', linestyle='--', alpha=0.7,
           label=f'500-tree floor: RMSE={rmse_by_ntrees[-1]:.4f}')
ax.set_xlabel('Number of trees'); ax.set_ylabel('Test RMSE ($100k)')
ax.set_title('RMSE vs Number of Trees — Variance Reduction Plateau', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Variance reduction plateaus after ~100 trees — adding more is rarely harmful, just slower.")

## XGBoost — Gradient Boosting

Each tree fits the **negative gradient of the loss** w.r.t. current predictions — for MSE, this is the residual:

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x}), \quad h_m \text{ fits } \{y_i - F_{m-1}(\mathbf{x}_i)\}$$

`early_stopping_rounds` halts training when validation RMSE stops improving — essential to prevent overfitting.

In [ ]:
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
    print("XGBoost available.")
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. Run: pip install xgboost")
    print("Remaining XGBoost cells will be skipped.")

In [ ]:
if XGB_AVAILABLE:
    # Hold out 15% of training data as early-stopping validation set
    X_tr2, X_val, y_tr2, y_val = train_test_split(
        X_train, y_tr, test_size=0.15, random_state=42)

    xgb = XGBRegressor(
        n_estimators=1000,        # upper bound — early stopping will find the right n
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        early_stopping_rounds=30,
        eval_metric='rmse',
        verbosity=0,
    )
    xgb.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)], verbose=False)

    y_pred_xgb = xgb.predict(X_test)
    rmse_xgb   = np.sqrt(mean_squared_error(y_te, y_pred_xgb))
    r2_xgb     = r2_score(y_te, y_pred_xgb)

    print(f"XGBoost:")
    print(f"  Best iteration:   {xgb.best_iteration}  (of 1000 rounds)")
    print(f"  Test RMSE: {rmse_xgb:.4f}  R²: {r2_xgb:.4f}")
else:
    print("Skipped — XGBoost not available.")
    y_pred_xgb, rmse_xgb, r2_xgb = None, None, None

In [ ]:
if XGB_AVAILABLE:
    # Learning curve: validation RMSE across boosting rounds
    val_rmse = xgb.evals_result()['validation_0']['rmse']

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(val_rmse, color='steelblue', linewidth=1.5)
    ax.axvline(xgb.best_iteration, color='coral', linestyle='--',
               label=f'Early stop at round {xgb.best_iteration}')
    ax.set_xlabel('Boosting round'); ax.set_ylabel('Validation RMSE')
    ax.set_title('XGBoost — Validation RMSE Across Boosting Rounds', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("Training past the early-stop point would overfit the validation signal.")
else:
    print("Skipped — XGBoost not available.")

## Full Regression Benchmark

Linear Regression (Ch.1 baseline) → Decision Tree → Random Forest → XGBoost

Each successive model adds complexity. Does it pay off in RMSE?

In [ ]:
lr = LinearRegression().fit(X_tr_sc, y_tr)
dt = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_tr)

model_results = {
    'Linear Regression':    (lr.predict(X_te_sc), 'needs scaling'),
    'Decision Tree (d=8)':  (dt.predict(X_test),  'no scaling'),
    'Random Forest (200)':  (y_pred_rf,            'no scaling'),
}
if XGB_AVAILABLE:
    model_results['XGBoost'] = (y_pred_xgb, 'no scaling')

print(f"{'Model':<28} {'RMSE':>8} {'R²':>8} {'Note':<20}")
print("-" * 65)
names, rmses, r2s = [], [], []
for name, (preds, note) in model_results.items():
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2   = r2_score(y_te, preds)
    print(f"{name:<28} {rmse:>8.4f} {r2:>8.4f}  {note}")
    names.append(name); rmses.append(rmse); r2s.append(r2)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#aec6e8', '#6baed6', '#2171b5', '#08306b'][:len(names)]

axes[0].bar(names, rmses, color=colors, edgecolor='white')
axes[0].set_ylabel('Test RMSE ($100k)'); axes[0].set_title('RMSE — lower is better', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(names, r2s, color=colors, edgecolor='white')
axes[1].set_ylabel('Test R²'); axes[1].set_title('R² — higher is better', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Regression Model Benchmark — California Housing', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## What Can Go Wrong — Boosting on Noisy Labels

In [ ]:
if XGB_AVAILABLE:
    # Inject 15% label noise into training targets
    rng = np.random.default_rng(42)
    y_tr_noisy = y_tr.copy()
    noise_idx  = rng.choice(len(y_tr_noisy), size=int(0.15 * len(y_tr_noisy)), replace=False)
    y_tr_noisy[noise_idx] += rng.normal(0, 2.0, len(noise_idx))   # add large random errors

    X_tr2n, X_valn, y_tr2n, y_valn = train_test_split(
        X_train, y_tr_noisy, test_size=0.15, random_state=42)

    # XGBoost WITHOUT early stopping — runs all 500 rounds
    xgb_no_es = XGBRegressor(n_estimators=500, learning_rate=0.05,
                               max_depth=4, random_state=42, verbosity=0)
    xgb_no_es.fit(X_tr2n, y_tr2n)
    rmse_no_es = np.sqrt(mean_squared_error(y_te, xgb_no_es.predict(X_test)))

    # XGBoost WITH early stopping
    xgb_es = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                            random_state=42, early_stopping_rounds=30,
                            eval_metric='rmse', verbosity=0)
    xgb_es.fit(X_tr2n, y_tr2n, eval_set=[(X_valn, y_valn)], verbose=False)
    rmse_es = np.sqrt(mean_squared_error(y_te, xgb_es.predict(X_test)))

    print("XGBoost trained on 15%-noisy labels:")
    print(f"  Without early stopping (500 rounds): RMSE = {rmse_no_es:.4f}")
    print(f"  With early stopping (stopped at {xgb_es.best_iteration}): RMSE = {rmse_es:.4f}")
    print(f"\nClean data XGBoost RMSE: {rmse_xgb:.4f}")
    print("Early stopping recovers much of the performance lost to noise overfitting.")
else:
    print("Skipped — XGBoost not available.")

## XGBoost Feature Importances vs Random Forest

In [ ]:
if XGB_AVAILABLE:
    xgb_importances = xgb.feature_importances_

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, imps, title in [
        (axes[0], rf_importances,   'Random Forest (200 trees)'),
        (axes[1], xgb_importances,  'XGBoost'),
    ]:
        order = np.argsort(imps)
        ax.barh([feature_names[i] for i in order], imps[order],
                color='steelblue', edgecolor='white')
        ax.set_xlabel('Importance'); ax.set_title(title, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

    plt.suptitle('Feature Importances — Random Forest vs XGBoost', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("Both agree MedInc is the dominant feature.")
    print("Rank differences reflect different importance definitions:")
    print("  RF:  mean Gini reduction across all trees")
    print("  XGB: default = 'weight' (split count); try importance_type='gain' for magnitude")
else:
    print("Skipped — XGBoost not available.")

## Exercises

1. **SVM C grid search:** sweep `C` over `[0.01, 0.1, 1, 10, 100]` using 5-fold cross-validation on the training set. Plot CV F1 vs C. How much does the best C improve on C=10?
2. **OOB vs test score:** for `n_estimators` in `[10, 20, 50, 100, 200, 500]`, plot both `oob_score_` and test R² on the same axis. When does OOB converge to test R²?
3. **XGBoost learning rate:** train XGBoost with `learning_rate` in `[0.3, 0.1, 0.05, 0.01]`, each with early stopping. Plot test RMSE vs best iteration for each learning rate. What is the trade-off?

In [ ]:
# Exercise 1: SVM C grid search
# TODO: from sklearn.model_selection import cross_val_score
# for C in [0.01, 0.1, 1, 10, 100]:
#     svm_cv = SVC(kernel='rbf', C=C, gamma='scale')
#     scores = cross_val_score(svm_cv, X_tr_sc, y_tr_cls, cv=5, scoring='f1')
#     print(f"C={C:6.2f}: mean F1={scores.mean():.4f}  std={scores.std():.4f}")

# Exercise 2: OOB vs test score across n_estimators
# TODO: for n in [10, 20, 50, 100, 200, 500]:
#     rf_n = RandomForestRegressor(n_estimators=n, oob_score=True, random_state=42)
#     rf_n.fit(X_train, y_tr)
#     record oob_score_ and r2_score(y_te, rf_n.predict(X_test))
# Plot both curves on same axis.

# Exercise 3: XGBoost learning rate effect
# TODO: for lr in [0.3, 0.1, 0.05, 0.01]:
#     xgb_lr = XGBRegressor(n_estimators=1000, learning_rate=lr, ...)
#     fit with early stopping; record test RMSE and best_iteration
# Plot table: lr | best_iteration | test RMSE